# Preentrenamiento con aves: Powdermill + Amazon Basin + PteroSet

Un caché aparte, `data/processed_extra`, con las mismas ventanas y mel del proyecto (3 s, 44,1 kHz,
128 bandas) pero con las cajas de tres datasets de aves anotados en Raven: Powdermill (Chronister et
al. 2021, Pensilvania, 48 especies), Amazon Basin (Hopping, Kahl y Klinck 2022, Madre de Dios, 132
especies; es el conjunto `PER` de BirdSet, pero la copia de HuggingFace pierde la banda de
frecuencia y por eso se baja de Zenodo) y PteroSet (Colombia, sin especie). Sirve para preentrenar
la cabeza del AST-DETR antes de afinarla con primates: `notebooks/pretrain_birds_train.py`. No toca
`src/`.

In [1]:
import json
import sys
import urllib.request
import zipfile
from collections.abc import Callable
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from matplotlib.axes import Axes
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

from core.config import SEED, P  # noqa: E402
from data import cache  # noqa: E402
from data.annotations import MIN_DURATION_S, load_annotations  # noqa: E402
from data.manifest import ClipWindow, build_manifest, duration_of, split_manifest  # noqa: E402
from data.raven import BOX_COLUMNS, CLEANED_BOX_COLUMNS, SPECIES  # noqa: E402
from data.species import LABEL_SEPARATOR, LabelSet  # noqa: E402
from prepare_data import build_dataset  # noqa: E402
from utils.audio import mel_db_range, mel_to_unit  # noqa: E402

POWDERMILL_DIR = Path("/data/fcandia/powdermill")
AMAZON_DIR = Path("/data/fcandia/amazon_basin")
PTEROSET_DIR = Path("/data/fcandia/pteroset/audio")
SOURCE_DIRS = {"powdermill": POWDERMILL_DIR, "amazon": AMAZON_DIR, "pteroset": PTEROSET_DIR}
PROCESSED_EXTRA = PROJECT_DIR / "data" / "processed_extra"
# Chronister et al. 2021 en Zenodo (CC0): WAV originales y tablas de Raven, como las de raw/
ZENODO_POWDERMILL = "https://zenodo.org/api/records/4656848/files"
POWDERMILL_FILES = ("README.txt", "annotation_Files.zip", "wav_Files.zip")
# Hopping, Kahl y Klinck 2022 en Zenodo (CC-BY 4.0): 21 FLAC de una hora a 32 kHz y un solo CSV
# con las cajas de todas las grabaciones, la especie como código de eBird
ZENODO_AMAZON = "https://zenodo.org/api/records/7079124/files"
AMAZON_FILES = ("annotations.csv", "species.csv", "soundscape_data.zip")
AMAZON_COLUMNS = dict(
    zip(["Start Time (s)", *BOX_COLUMNS[1:]], CLEANED_BOX_COLUMNS, strict=True)
)
# Como en prepare_data, una especie con menos cajas no es clase; pero acá va a `<fuente>/other`
# en vez de quedar fuera: ninguna vocalización anotada debe entrenarse como fondo.
MIN_PAIR_COUNT = 100
POW, AMZ, AV = "pow", "amz", "av"
OTHER = "other"
BIRD_CALL = f"{AV}{LABEL_SEPARATOR}voc"  # PteroSet: todas las cajas son AVEVOC, sin especie
EMPTY_RATIO = 0.25
SPLIT_RATIOS = (0.85, 0.15, 0.0)  # sin test: el test sigue siendo el de primates
SOURCES = {
    "primates": "#8a8a8a",
    "powdermill": "#2a78d6",
    "amazon": "#d9822b",
    "pteroset": "#1baf7a",
}
FORCE = False  # True regenera el caché aunque exista
pd.set_option("display.width", 140)

In [2]:
def ensure(base: str, directory: Path, names: tuple[str, ...], audio_suffix: str) -> None:
    # Baja lo que falte y descomprime los zips ahí mismo; con el audio ya en la carpeta el zip
    # no se vuelve a bajar.
    directory.mkdir(parents=True, exist_ok=True)
    for name in names:
        target = directory / name
        if target.is_file() or (
            name.endswith(".zip") and any(directory.rglob(f"*{audio_suffix}"))
        ):
            continue
        part = target.with_name(target.name + ".part")
        with urllib.request.urlopen(f"{base}/{name}/content") as response, open(part, "wb") as out:
            while chunk := response.read(1 << 20):
                out.write(chunk)
        part.rename(target)
        if name.endswith(".zip"):
            with zipfile.ZipFile(target) as archive:
                archive.extractall(directory)


for title, base, directory, names, suffix in (
    ("Powdermill", ZENODO_POWDERMILL, POWDERMILL_DIR, POWDERMILL_FILES, ".wav"),
    ("Amazon Basin", ZENODO_AMAZON, AMAZON_DIR, AMAZON_FILES, ".flac"),
):
    ensure(base, directory, names, suffix)
    recordings = sorted(p for p in directory.rglob("*") if p.suffix.lower() == suffix)
    info = sf.info(str(recordings[0]))
    print(
        f"{title}: {len(recordings)} grabaciones · {info.samplerate} Hz · {info.duration:.0f} s cada una"
    )

Powdermill: 77 grabaciones · 44100 Hz · 300 s cada una
Amazon Basin: 21 grabaciones · 32000 Hz · 3600 s cada una


In [3]:
def recording_of(table: Path) -> Path | None:
    # `X.Table.1.selections.txt` -> `X.wav` al lado; Powdermill mezcla `.wav` y `.WAV`.
    stem = table.name.split(".Table.")[0]
    for suffix in (".wav", ".WAV"):
        audio = table.with_name(stem + suffix)
        if audio.is_file():
            return audio
    return None


def tidy(df: pd.DataFrame) -> pd.DataFrame:
    # La misma higiene que `clean_annotations`: dentro del mel, con duración y ancho de banda
    df = df[["audio_path", *CLEANED_BOX_COLUMNS, "label"]].copy()
    df["low_freq_hz"] = df["low_freq_hz"].clip(lower=P.f_min)
    df["high_freq_hz"] = df["high_freq_hz"].clip(upper=P.f_max)
    duration = df["end_time_s"] - df["begin_time_s"]
    keep = (duration >= MIN_DURATION_S) & (df["high_freq_hz"] > df["low_freq_hz"])
    return df[keep].reset_index(drop=True)


def read_tables(root: Path, label_of: Callable[[pd.DataFrame], pd.Series | str]) -> pd.DataFrame:
    frames = []
    for table in sorted(root.rglob("*.selections.txt")):
        audio = recording_of(table)
        if audio is None:
            continue
        frame = pd.read_csv(table, sep="\t")
        frame = frame.rename(columns=dict(zip(BOX_COLUMNS, CLEANED_BOX_COLUMNS, strict=True)))
        frame["label"] = label_of(frame)
        frame["audio_path"] = str(audio)
        frames.append(frame)
    return tidy(pd.concat(frames, ignore_index=True))


def read_amazon() -> pd.DataFrame:
    # Un CSV para las 21 grabaciones; `????` es un ave sin identificar y va a `amz/other`.
    df = pd.read_csv(AMAZON_DIR / "annotations.csv").rename(columns=AMAZON_COLUMNS)
    code = df["Species eBird Code"].str.lower().replace("????", OTHER)
    df["label"] = AMZ + LABEL_SEPARATOR + code
    df["audio_path"] = [str(AMAZON_DIR / name) for name in df["Filename"]]
    return tidy(df)


def describe(name: str, df: pd.DataFrame) -> dict:
    # `duration_of` devuelve None en los wav ilegibles (raw/ tiene alguno): quedan fuera del total
    hours = sum(duration_of(path) or 0.0 for path in df["audio_path"].unique()) / 3600
    duration = df["end_time_s"] - df["begin_time_s"]
    return {
        "fuente": name,
        "grabaciones": df["audio_path"].nunique(),
        "horas": hours,
        "cajas": len(df),
        "etiquetas": df["label"].nunique(),
        "dur mediana (s)": duration.median(),
        "banda mediana (Hz)": (df["high_freq_hz"] - df["low_freq_hz"]).median(),
        "f máx p95 (Hz)": df["high_freq_hz"].quantile(0.95),
    }


powdermill = read_tables(POWDERMILL_DIR, lambda f: POW + LABEL_SEPARATOR + f[SPECIES].str.lower())
amazon = read_amazon()
pteroset = read_tables(PTEROSET_DIR, lambda f: BIRD_CALL)
primates = load_annotations()
primates = primates[primates["species"] != AV].copy()  # cleaned/ trae PteroSet como fondo
primates["label"] = primates["species"] + LABEL_SEPARATOR + primates["call_type"]
tables = {"primates": primates, "powdermill": powdermill, "amazon": amazon, "pteroset": pteroset}
pd.DataFrame([describe(name, df) for name, df in tables.items()]).set_index("fuente").round(2)

,grabaciones,horas,cajas,etiquetas,dur mediana (s),banda mediana (Hz),f máx p95 (Hz)
fuente,,,,,,,
primates,2664,24.85,19025,65,0.34,2281.04,14004.62
powdermill,76,6.33,16052,48,1.09,2573.20,9246.74
amazon,21,21.00,16480,133,2.50,1270.00,4645.00
pteroset,563,73.62,15371,1,0.82,2710.60,10334.10


In [4]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
for name, df in tables.items():
    duration = df["end_time_s"] - df["begin_time_s"]
    bandwidth = df["high_freq_hz"] - df["low_freq_hz"]
    style = {"histtype": "step", "lw": 1.6, "density": True, "color": SOURCES[name], "label": name}
    left.hist(duration, bins=np.logspace(-2, 1.3, 50), **style)
    right.hist(bandwidth, bins=np.logspace(2, 4.4, 50), **style)
left.set(xscale="log", xlabel="duración de la caja (s)", ylabel="densidad")
right.set(xscale="log", xlabel="ancho de banda de la caja (Hz)")
left.set_title("¿Se parecen las cajas? · duración")
right.set_title("ancho de banda")
for ax in (left, right):
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

In [5]:
def group_rare(df: pd.DataFrame, prefix: str) -> tuple[pd.DataFrame, int, int]:
    # -> (tabla con las especies escasas en `<prefix>/other`, especies que quedan, cajas en other)
    counts = df["label"].value_counts()
    frequent = counts[counts >= MIN_PAIR_COUNT].index
    other = prefix + LABEL_SEPARATOR + OTHER
    grouped = df.assign(label=df["label"].where(df["label"].isin(frequent), other))
    return grouped, int(frequent.difference([other]).size), int((grouped["label"] == other).sum())


powdermill, n_pow, other_pow = group_rare(powdermill, POW)
amazon, n_amz, other_amz = group_rare(amazon, AMZ)
extra = pd.concat([powdermill, amazon, pteroset], ignore_index=True)
labels = LabelSet(extra["label"])
print(
    f"{len(labels)} clases: {n_pow} especies de Powdermill y {n_amz} del Amazonas con >= "
    f"{MIN_PAIR_COUNT} cajas, {POW}/{OTHER} ({other_pow} cajas), {AMZ}/{OTHER} ({other_amz} cajas, "
    f"`????` incluidas) y {BIRD_CALL} ({len(pteroset)} cajas)"
)

per_class = extra["label"].value_counts().sort_values()
color_of = {POW: SOURCES["powdermill"], AMZ: SOURCES["amazon"], AV: SOURCES["pteroset"]}
fig, ax = plt.subplots(figsize=(8, 0.28 * len(per_class) + 1))
ax.barh(
    per_class.index,
    per_class.to_numpy(),
    color=[color_of[name.split(LABEL_SEPARATOR)[0]] for name in per_class.index],
)
ax.set(xscale="log", xlabel="cajas")
ax.axvline(MIN_PAIR_COUNT, color="black", lw=0.8, ls=":")
ax.grid(axis="x", alpha=0.3)
plt.show()

59 clases: 20 especies de Powdermill y 36 del Amazonas con >= 100 cajas, pow/other (608 cajas), amz/other (3919 cajas, `????` incluidas) y av/voc (15371 cajas)


In [6]:
manifest = build_manifest(extra, labels, empty_ratio=EMPTY_RATIO, seed=SEED)
train, val, test = split_manifest(manifest, n_classes=len(labels), ratios=SPLIT_RATIOS, seed=SEED)
train += test  # el tercer split existe por la firma de la función; acá no hay test
splits = {cache.TRAIN: train, cache.VAL: val, cache.TEST: []}


def source_of(window: ClipWindow) -> str:
    path = Path(window.audio_path)
    return next(name for name, directory in SOURCE_DIRS.items() if path.is_relative_to(directory))


def summarize(windows: list[ClipWindow]) -> dict:
    sources = pd.Series([source_of(w) for w in windows])
    return {
        "ventanas": len(windows),
        "con cajas": sum(len(w.boxes) > 0 for w in windows),
        "cajas": sum(len(w.boxes) for w in windows),
        "grabaciones": len({w.audio_path for w in windows}),
        **{f"ventanas {name}": int((sources == name).sum()) for name in SOURCE_DIRS},
    }


pd.DataFrame({name: summarize(windows) for name, windows in splits.items()}).T

,ventanas,con cajas,cajas,grabaciones,ventanas powdermill,ventanas amazon,ventanas pteroset
train,82360,62725,124763,478,10796,29803,41761
val,30871,22198,43844,182,3125,14850,12896
test,0,0,0,0,0,0,0


In [7]:
# Los .pt, labels.json, meta.json y *_sources.json van a processed_extra: se redirige el módulo
# `cache`, que es el único que conoce la carpeta, y `train.py` los lee por él.
cache.PROCESSED_DIR = PROCESSED_EXTRA
if (PROCESSED_EXTRA / "meta.json").exists() and not FORCE:
    print(f"ya existe {PROCESSED_EXTRA}; FORCE = True lo regenera")
else:
    PROCESSED_EXTRA.mkdir(parents=True, exist_ok=True)
    (PROCESSED_EXTRA / "meta.json").unlink(missing_ok=True)
    (PROCESSED_EXTRA / "labels.json").write_text(
        json.dumps(dict(enumerate(labels.names)), indent=2, ensure_ascii=False)
    )
    db_range = (0.0, 0.0)
    for name, windows in splits.items():
        cache.write_sources(name, windows)
        dataset = build_dataset(windows)
        if name == cache.TRAIN:
            db_range = mel_db_range(dataset["images"][:, 0])
        path = cache.split_path(name)
        torch.save(dataset, path)
        print(f"{name}: {len(windows)} ventanas -> {path} ({path.stat().st_size / 1024**3:.2f} GB)")
    meta = {
        "seed": SEED,
        "min_pair_count": MIN_PAIR_COUNT,
        "empty_ratio": EMPTY_RATIO,
        "split_ratios": SPLIT_RATIOS,
        "sources": {name: str(directory) for name, directory in SOURCE_DIRS.items()},
        "db_range": db_range,
        "params": asdict(P),
    }
    (PROCESSED_EXTRA / "meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))

train: 82360 ventanas -> /home/fcandia/tesis-primate/data/processed_extra/train.pt (13.04 GB)


val: 30871 ventanas -> /home/fcandia/tesis-primate/data/processed_extra/val.pt (4.89 GB)


test: 0 ventanas -> /home/fcandia/tesis-primate/data/processed_extra/test.pt (0.00 GB)


In [8]:
stored = torch.load(
    cache.split_path(cache.TRAIN), map_location="cpu", weights_only=False, mmap=True
)
db_low, db_high = cache.db_range()
sources = cache.sources(cache.TRAIN, len(stored["labels"]))


def draw_window(ax: Axes, index: int) -> None:
    image = mel_to_unit(stored["images"][index][0], db_low, db_high)
    ax.imshow(image, origin="lower", aspect="auto", cmap="magma", extent=(0, P.clip_len_s, 0, 1))
    for (cx, cy, w, h), class_id in zip(
        stored["boxes"][index].tolist(), stored["labels"][index].tolist(), strict=True
    ):
        x0, y0 = (cx - w / 2) * P.clip_len_s, cy - h / 2
        ax.add_patch(Rectangle((x0, y0), w * P.clip_len_s, h, fill=False, ec="white", lw=1.2))
        ax.text(x0, y0 + h, labels.name(class_id), color="white", fontsize=7, va="bottom")
    ax.set(xticks=[], yticks=[])


with_boxes = [i for i, boxes in enumerate(stored["boxes"]) if len(boxes)]
shown = np.random.default_rng(SEED).choice(with_boxes, size=9, replace=False)
fig, axes = plt.subplots(3, 3, figsize=(15, 8))
for ax, index in zip(axes.flat, shown.tolist(), strict=True):
    draw_window(ax, index)
    recording = Path(sources.recordings[sources.recording_of_window[index]])
    ax.set_title(f"{recording.name} · {sources.clip_start_s[index]:.1f} s", fontsize=8)
fig.legend(
    handles=[Line2D([], [], color="white", lw=1.2, label="caja anotada")],
    loc="upper center",
    facecolor="#303030",
    labelcolor="white",
    fontsize=9,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

In [9]:
print(
    "Preentrenar (caché processed_extra, corridas en runs_extra/):\n"
    "  uv run python notebooks/pretrain_birds_train.py --name detr_birds_v2 --device cuda:1\n"
    "Afinar con primates desde ese checkpoint (caché y runs/ normales, cabezas de clase nuevas):\n"
    "  uv run python notebooks/pretrain_birds_train.py --finetune runs_extra/detr_birds_v2/best.pt "
    "--name detr_t10_logmel_birds_v3 --device cuda:1"
)

Preentrenar (caché processed_extra, corridas en runs_extra/):
  uv run python notebooks/pretrain_birds_train.py --name detr_birds_v2 --device cuda:1
Afinar con primates desde ese checkpoint (caché y runs/ normales, cabezas de clase nuevas):
  uv run python notebooks/pretrain_birds_train.py --finetune runs_extra/detr_birds_v2/best.pt --name detr_t10_logmel_birds_v3 --device cuda:1
